In [ ]:
!pip install -q qiskit qiskit-machine-learning kagglehub torchaudio soundfile torchvision torchlibrosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 4.5 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split

from kagglehub import dataset_download
from sklearn.metrics import classification_report, confusion_matrix

import soundfile as sf
import torchaudio

from qiskit.circuit.library import ZZFeatureMap, real_amplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

In [ ]:
BATCH_SIZE = 8
FEATURE_DIM = 8           # <= 8 is sensible
EPOCHS = 40
LR = 1e-4

TARGET_SR = 16000         # wav2vec2 expects 16kHz
CLIP_SECONDS = 4.0
FIXED_LEN = int(TARGET_SR * CLIP_SECONDS)

torch.manual_seed(42)
np.random.seed(42)

device_cnn = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_qnn = torch.device("cpu")   # keep QNN on CPU
print("CNN device:", device_cnn, "| QNN device:", device_qnn)

CNN device: cuda | QNN device: cpu


In [ ]:
dataset_root = dataset_download("mazinrabie/heart-sound")
data_path = os.path.join(dataset_root, "Dataset")
print("Top folders:", os.listdir(data_path))

TARGET_CLASSES = [
    'Mitral valve prolapse',
    'Normal',
    'aortic stenosis',
    'mitral stenosis',
    'mitral regurgitation'
]
CLASS_TO_IDX = {c: i for i, c in enumerate(TARGET_CLASSES)}
NUM_CLASSES = len(TARGET_CLASSES)

Using Colab cache for faster access to the 'heart-sound' dataset.
Top folders: ['Mitral valve prolapse', 'Normal', 'aortic stenosis', 'mitral stenosis', 'mitral regurgitation']


In [ ]:
def load_wav_fixed(path, target_sr=TARGET_SR, fixed_len=FIXED_LEN):
    """Load audio, mono, resample to 16k, normalize, pad/truncate to fixed length. Returns [T]."""
    wav_np, sr = sf.read(path)
    wav = torch.tensor(wav_np).float()

    if wav.ndim == 2:  # stereo -> mono
        wav = wav.mean(dim=1)

    wav = wav.unsqueeze(0)  # [1, T]

    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=target_sr)

    # normalize amplitude
    wav = wav / (wav.abs().max() + 1e-9)

    T = wav.shape[-1]
    if T < fixed_len:
        wav = torch.nn.functional.pad(wav, (0, fixed_len - T))
    else:
        wav = wav[:, :fixed_len]

    return wav.squeeze(0)  # [T]

In [ ]:
from collections import defaultdict
import os
import random
from collections import defaultdict
from torch.utils.data import Dataset

class AudioWaveformDataset(Dataset):
    def __init__(self, root, max_per_class=100, seed=42):
        self.samples, self.labels = [], []
        self.class_counts = defaultdict(int)

        random.seed(seed)

        for class_name in TARGET_CLASSES:
            class_dir = os.path.join(root, class_name)
            if not os.path.isdir(class_dir):
                raise RuntimeError(f"Missing folder: {class_dir}")

            # collect all wav files
            wav_files = [
                f for f in os.listdir(class_dir)
                if f.lower().endswith(".wav")
            ]

            if len(wav_files) < max_per_class:
                raise RuntimeError(
                    f"Class '{class_name}' has only {len(wav_files)} files (< {max_per_class})"
                )

            # 🔥 random but reproducible downsampling
            selected_files = random.sample(wav_files, max_per_class)

            for fname in selected_files:
                self.samples.append(os.path.join(class_dir, fname))
                self.labels.append(CLASS_TO_IDX[class_name])
                self.class_counts[class_name] += 1

        if len(self.samples) == 0:
            raise RuntimeError("No audio files found!")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        wav = load_wav_fixed(self.samples[idx])
        y = self.labels[idx]
        return wav, y

# class AudioWaveformDataset(Dataset):
#     def __init__(self, root):
#         self.samples, self.labels = [], []
#         self.class_counts = defaultdict(int)

#         for class_name in TARGET_CLASSES:
#             class_dir = os.path.join(root, class_name)
#             if not os.path.isdir(class_dir):
#                 raise RuntimeError(f"Missing folder: {class_dir}")

#             for fname in os.listdir(class_dir):
#                 if fname.lower().endswith(".wav"):
#                     self.samples.append(os.path.join(class_dir, fname))
#                     self.labels.append(CLASS_TO_IDX[class_name])
#                     self.class_counts[class_name] += 1

#         if len(self.samples) == 0:
#             raise RuntimeError("No audio files found! Check folder structure/classes.")

#     def __len__(self):
#         return len(self.samples)

#     def __getitem__(self, idx):
#         wav = load_wav_fixed(self.samples[idx])  # [T]
#         y = self.labels[idx]
#         return wav, y

full_dataset = AudioWaveformDataset(data_path,max_per_class=50,seed=42)
#full_dataset = AudioWaveformDataset(data_path)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
generator = torch.Generator().manual_seed(42)

train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=generator)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print("Disease Classes:")
for i, c in enumerate(TARGET_CLASSES):
    print(f"  {i}: {c}")

print("\nFiles per class:")
for class_name in TARGET_CLASSES:
    print(f"  {class_name}: {full_dataset.class_counts[class_name]}")


Disease Classes:
  0: Mitral valve prolapse
  1: Normal
  2: aortic stenosis
  3: mitral stenosis
  4: mitral regurgitation

Files per class:
  Mitral valve prolapse: 50
  Normal: 50
  aortic stenosis: 50
  mitral stenosis: 50
  mitral regurgitation: 50


In [ ]:
feature_map = ZZFeatureMap(feature_dimension=FEATURE_DIM, reps=1)
ansatz = real_amplitudes(num_qubits=FEATURE_DIM, reps=1)

qc = QuantumCircuit(FEATURE_DIM)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)

observables = []
for i in range(FEATURE_DIM):
    pauli = ["I"] * FEATURE_DIM
    pauli[i] = "Z"
    observables.append(SparsePauliOp("".join(pauli)))

qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
    observables=observables
)

/tmp/ipython-input-1096402979.py:1: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=FEATURE_DIM, reps=1)


In [ ]:
qc.decompose().draw()

┌───┐┌───────────┐                                             »
q_0: ┤ H ├┤ P(2*x[0]) ├──■──────────────────────────────────■────■──»
     ├───┤├───────────┤┌─┴─┐┌────────────────────────────┐┌─┴─┐  │  »
q_1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├┤ X ├──┼──»
     ├───┤├───────────┤└───┘└────────────────────────────┘└───┘┌─┴─┐»
q_2: ┤ H ├┤ P(2*x[2]) ├────────────────────────────────────────┤ X ├»
     ├───┤├───────────┤                                        └───┘»
q_3: ┤ H ├┤ P(2*x[3]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_4: ┤ H ├┤ P(2*x[4]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_5: ┤ H ├┤ P(2*x[5]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_6: ┤ H ├┤ P(2*x[6]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_7: ┤ H ├┤ P(2*x[7]) ├─────────────────────────────────────────────»
     └───┘└───────────┘                                             »
«                                                  »
«q_0: ────────────────────────────────■─────────■──»
«                                     │         │  »
«q_1: ────────────────────────────────┼────■────┼──»
«     ┌────────────────────────────┐┌─┴─┐┌─┴─┐  │  »
«q_2: ┤ P(2*(π - x[0])*(π - x[2])) ├┤ X ├┤ X ├──┼──»
«     └────────────────────────────┘└───┘└───┘┌─┴─┐»
«q_3: ────────────────────────────────────────┤ X ├»
«                                             └───┘»
«q_4: ─────────────────────────────────────────────»
«                                                  »
«q_5: ─────────────────────────────────────────────»
«                                                  »
«q_6: ─────────────────────────────────────────────»
«                                                  »
«q_7: ─────────────────────────────────────────────»
«                                                  »
«                                                       »
«q_0: ─────────────────────────────────────■─────────■──»
«                                          │         │  »
«q_1: ────────────────────────────────■────┼────■────┼──»
«     ┌────────────────────────────┐┌─┴─┐  │    │    │  »
«q_2: ┤ P(2*(π - x[1])*(π - x[2])) ├┤ X ├──┼────┼────┼──»
«     ├────────────────────────────┤└───┘┌─┴─┐┌─┴─┐  │  »
«q_3: ┤ P(2*(π - x[0])*(π - x[3])) ├─────┤ X ├┤ X ├──┼──»
«     └────────────────────────────┘     └───┘└───┘┌─┴─┐»
«q_4: ─────────────────────────────────────────────┤ X ├»
«                                                  └───┘»
«q_5: ──────────────────────────────────────────────────»
«                                                       »
«q_6: ──────────────────────────────────────────────────»
«                                                       »
«q_7: ──────────────────────────────────────────────────»
«                                                       »
«                                                            »
«q_0: ─────────────────────────────────────■──────────────■──»
«                                          │              │  »
«q_1: ────────────────────────────────■────┼─────────■────┼──»
«                                     │    │         │    │  »
«q_2: ────────────────────────────────┼────┼────■────┼────┼──»
«     ┌────────────────────────────┐┌─┴─┐  │  ┌─┴─┐  │    │  »
«q_3: ┤ P(2*(π - x[1])*(π - x[3])) ├┤ X ├──┼──┤ X ├──┼────┼──»
«     ├────────────────────────────┤└───┘┌─┴─┐└───┘┌─┴─┐  │  »
«q_4: ┤ P(2*(π - x[0])*(π - x[4])) ├─────┤ X ├─────┤ X ├──┼──»
«     └────────────────────────────┘     └───┘     └───┘┌─┴─┐»
«q_5: ──────────────────────────────────────────────────┤ X ├»
«                                                       └───┘»
«q_6: ───────────────────────────────────────────────────────»
«                                             

In [ ]:
bundle = torchaudio.pipelines.WAV2VEC2_BASE
wav2vec = bundle.get_model().to(device_cnn)
wav2vec.eval()

for p in wav2vec.parameters():
    p.requires_grad = False

Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960.pth


100%|██████████| 360M/360M [00:00<00:00, 438MB/s]


In [ ]:
class HybridWav2VecHQCNN(nn.Module):
    def __init__(self, wav2vec, qnn, feature_dim, num_classes):
        super().__init__()
        self.wav2vec = wav2vec

        self.reduce = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Linear(128, feature_dim),
            nn.Tanh()
        )

        self.qnn = TorchConnector(qnn)
        self.fc = nn.Linear(feature_dim, num_classes)   # 🔥 now 8 -> classes

    def forward(self, wav):
        # wav: [B, T]
        wav = wav.to(device_cnn)

        with torch.no_grad():
            features, _ = self.wav2vec(wav)             # [B, frames, 768]

        emb = features.mean(dim=1)                      # [B, 768]
        x = self.reduce(emb)                            # [B, 8]

        x = x.to(device_qnn)
        x = self.qnn(x)                                 # [B, 8]  ✅ multi-output
        logits = self.fc(x)                             # [B, num_classes]
        return logits

In [ ]:
model = HybridWav2VecHQCNN(wav2vec, qnn, FEATURE_DIM, NUM_CLASSES)

model.reduce.to(device_cnn)
model.qnn.to(device_qnn)
model.fc.to(device_qnn)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

In [ ]:
best_val_acc = -1.0

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for wavs, labels in train_loader:
        labels = torch.as_tensor(labels, dtype=torch.long, device=device_qnn)

        optimizer.zero_grad()
        logits = model(wavs)                 # logits on CPU
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.numel()

    train_acc = 100.0 * correct / max(total, 1)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Acc: {train_acc:.2f}%")

    # quick val acc each epoch (optional but useful)
    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for wavs, labels in val_loader:
            labels = torch.as_tensor(labels, dtype=torch.long, device=device_qnn)
            logits = model(wavs)
            preds = logits.argmax(dim=1)
            v_correct += (preds == labels).sum().item()
            v_total += labels.numel()
    val_acc = 100.0 * v_correct / max(v_total, 1)
    print(f"         Val Acc: {val_acc:.2f}%")

    # checkpoint (TorchConnector-safe)
    ckpt = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "meta": {
            "feature_dim": FEATURE_DIM,
            "num_classes": NUM_CLASSES,
            "target_sr": TARGET_SR,
            "clip_seconds": CLIP_SECONDS,
            "epoch": epoch + 1,
            "val_acc": val_acc
        }
    }
    torch.save(ckpt, "heartsound_hybrid_wav2vec_qnn_checkpoint.pt")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(ckpt, "heartsound_hybrid_wav2vec_qnn_best.pt")
        print("         Saved best checkpoint.")

print("Training done. Best Val Acc:", best_val_acc)

Epoch 1/40 | Loss: 1.6100 | Acc: 23.50%
         Val Acc: 12.00%
         Saved best checkpoint.
Epoch 2/40 | Loss: 1.6079 | Acc: 28.50%
         Val Acc: 12.00%
Epoch 3/40 | Loss: 1.6079 | Acc: 27.50%
         Val Acc: 14.00%
         Saved best checkpoint.
Epoch 4/40 | Loss: 1.6104 | Acc: 26.50%
         Val Acc: 14.00%
Epoch 5/40 | Loss: 1.6068 | Acc: 26.00%
         Val Acc: 14.00%
Epoch 6/40 | Loss: 1.6098 | Acc: 26.00%
         Val Acc: 14.00%
Epoch 7/40 | Loss: 1.6074 | Acc: 26.00%
         Val Acc: 14.00%
Epoch 8/40 | Loss: 1.6082 | Acc: 27.00%
         Val Acc: 12.00%
Epoch 9/40 | Loss: 1.6075 | Acc: 26.50%
         Val Acc: 12.00%
Epoch 10/40 | Loss: 1.6078 | Acc: 28.00%
         Val Acc: 12.00%
Epoch 11/40 | Loss: 1.6073 | Acc: 27.00%
         Val Acc: 12.00%
Epoch 12/40 | Loss: 1.6074 | Acc: 27.00%
         Val Acc: 12.00%
Epoch 13/40 | Loss: 1.6077 | Acc: 26.00%
         Val Acc: 12.00%
Epoch 14/40 | Loss: 1.6102 | Acc: 25.50%
         Val Acc: 12.00%
Epoch 15/40 | Loss: 1

In [ ]:
torch.save(ckpt, "heartsound_hybrid_wav2vec_qnn_best.pt")

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for wavs, labels in val_loader:
        logits = model(wavs)
        preds = logits.argmax(dim=1)
        y_true.extend(np.array(labels))
        y_pred.extend(preds.cpu().numpy())

labels_list = list(range(NUM_CLASSES))
print(classification_report(y_true, y_pred, labels=labels_list, target_names=TARGET_CLASSES, zero_division=0))
print(confusion_matrix(y_true, y_pred, labels=labels_list))